## INM705 Deep Learning for Image Analysis
#### Object detection with YOLO
#### Author: Bo Fu, Yehoshua Perez Condori

In [3]:
# 20 classes
CLASSES = ['aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor']

CLASS2IDX = {cls: idx for idx, cls in enumerate(CLASSES)}
print(CLASS2IDX)

{'aeroplane': 0, 'bicycle': 1, 'bird': 2, 'boat': 3, 'bottle': 4, 'bus': 5, 'car': 6, 'cat': 7, 'chair': 8, 'cow': 9, 'diningtable': 10, 'dog': 11, 'horse': 12, 'motorbike': 13, 'person': 14, 'pottedplant': 15, 'sheep': 16, 'sofa': 17, 'train': 18, 'tvmonitor': 19}


In [4]:
# Set YOLO parameters
S = 7    # 7×7 grid size
B = 2    # number of boxes per cell
C = 20   # number of classes

In [5]:
from torchvision import transforms
from PIL import Image

# define image transform pipeline
transform = transforms.Compose([
    transforms.Resize((448, 448)),        # resize image to 448x448
    transforms.ToTensor(),                # convert PIL image to tensor, pixel values 0~1
    transforms.Normalize(                 # normalize using ImageNet mean and std
        mean=[0.485, 0.456, 0.406],       # mean for R, G, B channels
        std=[0.229, 0.224, 0.225]         # std for R, G, B channels
    )
])


def load_image(img_path):
    # open image file
    img = Image.open(img_path)

    # convert to RGB (some images might be grayscale or RGBA)
    img = img.convert('RGB')

    # apply transform: resize + to tensor + normalize
    img = transform(img)

    # return tensor of shape (3, 448, 448)
    return img






# test
if __name__ == '__main__':
    
    img_path = 'data/VOC2012/JPEGImages/2008_000008.jpg'
    
    img = load_image(img_path)
    
    print("Image shape:", img.shape)   # (3, 448, 448)
    print("Min value:",  img.min())    # around -2.0 after normalize
    print("Max value:",  img.max())    # around 2.0 after normalize

Image shape: torch.Size([3, 448, 448])
Min value: tensor(-2.1179)
Max value: tensor(2.6400)


In [6]:
import xml.etree.ElementTree as ET
import torch

def parse_xml(xml_path):
    # read and parse the XML file
    tree = ET.parse(xml_path)
    root = tree.getroot()

    # get original image width and height
    # used to normalize coordinates to 0~1
    w = float(root.find('size/width').text)
    h = float(root.find('size/height').text)

    boxes  = []  # store bounding boxes
    labels = []  # store class labels

    # loop through each object in the XML
    for obj in root.findall('object'):

        # get class name, e.g. 'dog', 'car'
        name = obj.find('name').text.strip()

        # only process if class is in our 20 classes
        if name in CLASS2IDX:

            # get bounding box coordinates (pixel values)
            b  = obj.find('bndbox')
            x1 = float(b.find('xmin').text)
            y1 = float(b.find('ymin').text)
            x2 = float(b.find('xmax').text)
            y2 = float(b.find('ymax').text)
    
            # normalize coordinates to 0~1
            x1 = x1 / w
            y1 = y1 / h
            x2 = x2 / w
            y2 = y2 / h
    
            # add to lists
            boxes.append([x1, y1, x2, y2])
            labels.append(CLASS2IDX[name])

    # if no objects found, return empty tensors
    if len(boxes) == 0:
        return torch.zeros((0, 4)), torch.zeros((0,), dtype=torch.long)

    # convert lists to tensors
    boxes  = torch.tensor(boxes,  dtype=torch.float32)  # shape (N, 4)
    labels = torch.tensor(labels, dtype=torch.long)      # shape (N,)

    return boxes, labels






# test
if __name__ == '__main__':
    xml_path = 'data/VOC2012/Annotations/2008_000008.xml'
    boxes, labels = parse_xml(xml_path)
    
    print("boxes:",  boxes)    # tensor of coordinates
    print("labels:", labels)   # tensor of class indices
    print("number of objects:", len(boxes))

boxes: tensor([[0.1060, 0.1968, 0.9420, 0.9502],
        [0.3160, 0.0995, 0.5780, 0.3778]])
labels: tensor([12, 14])
number of objects: 2


In [7]:
def encode_yolo(boxes, labels):
    # input:
    #   boxes  : (N, 4)  normalized x1,y1,x2,y2
    #   labels : (N,)    class index
    # output:
    #   target : (7, 7, 30)

    # initialize all zeros
    # shape: (S, S, B*5 + C) = (7, 7, 30)
    target = torch.zeros((S, S, B * 5 + C))

    # loop through each object
    for i in range(len(boxes)):

        # get coordinates
        x1, y1, x2, y2 = boxes[i]

        # get class index
        cls_idx = labels[i].item()

        # calculate center point
        cx = (x1 + x2) / 2.0
        cy = (y1 + y2) / 2.0

        # calculate width and height
        w = x2 - x1
        h = y2 - y1

        # find which grid cell the center falls into
        gx = int(cx * S)       # column index 0~6
        gy = int(cy * S)       # row index 0~6

        # prevent out of bounds
        gx = min(gx, S - 1)
        gy = min(gy, S - 1)

        # if cell is empty, fill in the object
        if target[gy, gx, 4] == 0.0:

            # calculate offset inside the cell (0~1)
            x_cell = cx * S - gx
            y_cell = cy * S - gy

            # fill box 1 values
            target[gy, gx, 0] = x_cell   # x offset in cell
            target[gy, gx, 1] = y_cell   # y offset in cell
            target[gy, gx, 2] = w        # width
            target[gy, gx, 3] = h        # height
            target[gy, gx, 4] = 1.0      # confidence = 1

            # fill class label (one-hot encoding)
            target[gy, gx, B * 5 + cls_idx] = 1.0

    return target




# test
if __name__ == '__main__':
    xml_path = 'data/VOC2012/Annotations/2008_000008.xml'
    boxes, labels = parse_xml(xml_path)
    
    target = encode_yolo(boxes, labels)
    
    print("target shape:", target.shape)          # (7, 7, 30)
    print("cells with object:", (target[..., 4] > 0).sum().item())

target shape: torch.Size([7, 7, 30])
cells with object: 2


In [8]:
from torch.utils.data import Dataset


class VOCDataset(Dataset):

    def __init__(self, txt_file):
        # read image ID list from txt file
        with open(txt_file, 'r') as f:
            self.img_ids = [line.strip() for line in f.readlines()]

    def __len__(self):
        # return total number of images
        return len(self.img_ids)

    def __getitem__(self, idx):
        # get image ID e.g. '2008_000008'
        img_id = self.img_ids[idx]

        # load image
        img = load_image(f'{IMG_DIR}/{img_id}.jpg')

        # parse XML and encode
        boxes, labels = parse_xml(f'{ANN_DIR}/{img_id}.xml')
        target = encode_yolo(boxes, labels)

        return img, target



# paths
ROOT     = 'data/VOC2012'
IMG_DIR  = 'data/VOC2012/JPEGImages'
ANN_DIR  = 'data/VOC2012/Annotations'
TRAIN_TXT = 'data/VOC2012/ImageSets/Main/train.txt'
VAL_TXT   = 'data/VOC2012/ImageSets/Main/val.txt'



# test
if __name__ == '__main__':
    dataset = VOCDataset(TRAIN_TXT)
    
    print("Total images:", len(dataset))
    
    img, target = dataset[0]
    print("img shape:   ", img.shape)
    print("target shape:", target.shape)

Total images: 5717
img shape:    torch.Size([3, 448, 448])
target shape: torch.Size([7, 7, 30])


In [9]:
from torch.utils.data import DataLoader


def get_dataloaders(batch_size=16):

    # create train and val datasets
    train_dataset = VOCDataset(TRAIN_TXT)
    val_dataset   = VOCDataset(VAL_TXT)

    # create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,  # load 16 images at a time
        shuffle=True,           # shuffle every epoch
        num_workers=0           # set 0 for local machine
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,          # no shuffle for validation
        num_workers=0
    )

    print("Train set:", len(train_dataset), "images")
    print("Val set:  ", len(val_dataset),   "images")

    return train_loader, val_loader


# test
if __name__ == '__main__':
    train_loader, val_loader = get_dataloaders()

    # get one batch
    imgs, targets = next(iter(train_loader))

    print("imgs shape:   ", imgs.shape)     # (16, 3, 448, 448)
    print("targets shape:", targets.shape)  # (16, 7, 7, 30)

Train set: 5717 images
Val set:   5823 images
imgs shape:    torch.Size([16, 3, 448, 448])
targets shape: torch.Size([16, 7, 7, 30])
